# Mini-Project: Advanced Statistical Analysis of Apple Inc. Stock Data

This notebook uses Apple Inc. (AAPL) historical stock data only and directly addresses the mini-project requirements:
1. Load and explore the AAPL time-series dataset
2. Plot closing prices over time
3. Build a candlestick chart
4. Calculate and visualize moving averages
5. Perform a t-test comparing closing prices between different years
6. Summarize key findings

## 1. Data Loading and Exploration
We import the required libraries, download the dataset if needed, and inspect the time-series structure.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle

from scipy import stats
from scipy.signal import savgol_filter

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

DATA_URL = 'https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%203/W3D4%20-%20Mini%20Project/Apple%20Stock%20Prices%20From%201981%20to%202023.zip'
DATA_DIR = Path('data')
ZIP_PATH = DATA_DIR / 'apple_stock_prices.zip'
EXTRACT_DIR = DATA_DIR / 'apple_stock_prices'

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.exists():
    urlretrieve(DATA_URL, ZIP_PATH)

if not EXTRACT_DIR.exists():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

csv_files = sorted(EXTRACT_DIR.rglob('*.csv'))
if not csv_files:
    raise FileNotFoundError('No CSV file found after extraction.')

apple_candidates = [p for p in csv_files if 'apple' in p.name.lower() or 'aapl' in p.name.lower()]
csv_path = apple_candidates[0] if apple_candidates else csv_files[0]
print(f'Using dataset: {csv_path}')

In [ ]:
df = pd.read_csv(csv_path)

# Standardize column names for easier processing.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('.', '', regex=False)
)

required_cols = {'date', 'open', 'high', 'low', 'close', 'volume'}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').set_index('date')

print('Dataset check: Apple stock time-series with OHLCV columns loaded successfully.')
display(df.head())
display(df.tail())

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nNull values per column:')
print(df.isna().sum())

start_date = df.index.min()
end_date = df.index.max()
inferred_freq = pd.infer_freq(df.index[:40])
rows_per_year = df.groupby(df.index.year).size()

print('\nTime-series overview:')
print(f'- Start date: {start_date.date()}')
print(f'- End date: {end_date.date()}')
print(f'- Number of rows: {len(df)}')
print(f'- Inferred frequency on early sample: {inferred_freq}')
print(f'- Average trading days per year: {rows_per_year.mean():.1f}')

print('\nRequirement checklist:')
print('- [x] Apple stock dataset loaded')
print('- [x] Closing price time-series ready')
print('- [x] Candlestick inputs (Open/High/Low/Close) available')
print('- [x] Moving averages can be computed')
print('- [x] Data grouped by year for t-test')

## 2. Data Visualization
We plot closing prices over time, then build a candlestick chart for recent trading days.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

axes[0].plot(df.index, df['close'], color='tab:blue', linewidth=1.3)
axes[0].set_title('AAPL Closing Price Over Time')
axes[0].set_ylabel('Close Price (USD)')

axes[1].plot(df.index, df['volume'], color='tab:orange', linewidth=1.0)
axes[1].set_title('AAPL Traded Volume Over Time')
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Candlestick chart for the last 180 trading days using only Matplotlib.
recent = df[['open', 'high', 'low', 'close']].tail(180).copy()
recent['date_num'] = mdates.date2num(recent.index.to_pydatetime())

fig, ax = plt.subplots(figsize=(14, 6))
bar_width = 0.6

for idx, row in recent.iterrows():
    x = mdates.date2num(idx.to_pydatetime())
    o, h, l, c = row['open'], row['high'], row['low'], row['close']
    color = '#2ca02c' if c >= o else '#d62728'

    ax.vlines(x, l, h, color=color, linewidth=1.0)

    body_low = min(o, c)
    body_height = max(abs(c - o), 1e-3)
    rect = Rectangle((x - bar_width / 2, body_low), bar_width, body_height,
                     facecolor=color, edgecolor=color, alpha=0.8)
    ax.add_patch(rect)

ax.set_title('AAPL Candlestick Chart (Last 180 Trading Days)')
ax.set_ylabel('Price (USD)')
ax.set_xlabel('Date')
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()

plt.tight_layout()
plt.show()

## 3. Statistical Analysis
We compute summary statistics and analyze the closing price trend with moving averages.

In [ ]:
key_cols = [col for col in ['open', 'high', 'low', 'close', 'adj_close', 'volume'] if col in df.columns]
summary_stats = df[key_cols].agg(['mean', 'median', 'std']).T
summary_stats = summary_stats.rename(columns={'std': 'standard_deviation'})
display(summary_stats)

In [ ]:
df['ma_20'] = df['close'].rolling(window=20).mean()
df['ma_50'] = df['close'].rolling(window=50).mean()

# NumPy convolution-based moving average (20 days).
window = 20
weights = np.ones(window) / window
ma_conv = np.convolve(df['close'].values, weights, mode='valid')
ma_conv_series = pd.Series(ma_conv, index=df.index[window - 1:], name='ma_20_convolve')

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df['close'], label='Close', linewidth=1.0, alpha=0.7)
ax.plot(df.index, df['ma_20'], label='MA 20 (rolling)', linewidth=1.8)
ax.plot(df.index, df['ma_50'], label='MA 50 (rolling)', linewidth=1.8)
ax.plot(ma_conv_series.index, ma_conv_series.values, label='MA 20 (np.convolve)', linewidth=1.5, linestyle='--')
ax.set_title('Closing Price with Moving Averages')
ax.set_ylabel('Price (USD)')
ax.set_xlabel('Date')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Hypothesis Testing
We compare average closing prices between two years with a t-test.

In [ ]:
yearly_counts = df.groupby(df.index.year).size()
eligible_years = yearly_counts[yearly_counts >= 200].index.tolist()

if len(eligible_years) < 2:
    raise ValueError('Not enough full years with >=200 observations for t-test.')

year_a = min(eligible_years)
year_b = max(eligible_years)

sample_a = df.loc[df.index.year == year_a, 'close'].dropna()
sample_b = df.loc[df.index.year == year_b, 'close'].dropna()

t_stat, p_value = stats.ttest_ind(sample_a, sample_b, equal_var=False, nan_policy='omit')

print(f'Comparing average close prices: {year_a} vs {year_b}')
print(f'- n({year_a}) = {len(sample_a)} | mean = {sample_a.mean():.2f}')
print(f'- n({year_b}) = {len(sample_b)} | mean = {sample_b.mean():.2f}')
print(f'- t-statistic = {t_stat:.4f}')
print(f'- p-value = {p_value:.6f}')

if p_value < 0.05:
    print('Result: reject H0 at 5% level (means are significantly different).')
else:
    print('Result: fail to reject H0 at 5% level (no significant difference detected).')

In [ ]:
# Optional extension: inspect daily return distribution after the required t-test.
df['daily_return'] = df['close'].pct_change()
returns = df['daily_return'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(returns, bins=80, color='tab:blue', alpha=0.8, edgecolor='black')
axes[0].set_title('Daily Returns Distribution')
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')

stats.probplot(returns, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot vs Normal Distribution')

plt.tight_layout()
plt.show()

normaltest_stat, normaltest_p = stats.normaltest(returns)
print(f'Normality test p-value: {normaltest_p:.6e}')

## 5. Advanced Statistical Techniques (Bonus)
This optional section uses NumPy and SciPy tools for deeper insight beyond the required tasks.

In [ ]:
# Correlation between moving averages and volume.
df['ma_10'] = df['close'].rolling(10).mean()
df['ma_30'] = df['close'].rolling(30).mean()

corr_df = df[['ma_10', 'ma_20', 'ma_30', 'volume']].dropna()
corr_matrix = corr_df.corr()
display(corr_matrix)

corr_ma20_volume = np.corrcoef(corr_df['ma_20'], corr_df['volume'])[0, 1]
print(f'NumPy corrcoef(ma_20, volume): {corr_ma20_volume:.4f}')

rolling_corr = corr_df['ma_20'].rolling(60).corr(corr_df['volume'])
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rolling_corr.index, rolling_corr.values, color='tab:purple', linewidth=1.2)
ax.axhline(0.0, color='black', linewidth=1.0, linestyle='--')
ax.set_title('60-Day Rolling Correlation: MA 20 vs Volume')
ax.set_ylabel('Correlation')
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# Signal processing with SciPy: Savitzky-Golay smoothing of close prices.
close_values = df['close'].values
window_length = 31 if len(close_values) >= 31 else len(close_values) - 1
if window_length % 2 == 0:
    window_length -= 1

if window_length >= 5:
    smooth_close = savgol_filter(close_values, window_length=window_length, polyorder=3)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df.index, close_values, label='Original Close', alpha=0.5)
    ax.plot(df.index, smooth_close, label='Savitzky-Golay Smoothed', linewidth=2.0)
    ax.set_title('Signal Processing on Closing Price (SciPy)')
    ax.set_ylabel('Price (USD)')
    ax.set_xlabel('Date')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Dataset too short for Savitzky-Golay smoothing with safe parameters.')

## 6. Summary and Insights
- The notebook uses Apple stock OHLCV time-series data and not a classification dataset.
- Closing prices over time show long-run trend shifts and strong growth periods.
- The candlestick chart highlights short-term price action and daily volatility.
- Moving averages (20 and 50 days) smooth short-term noise and reveal medium-term trend direction.
- The two-sample t-test compares mean closing prices across different years and reports statistical significance.
- Optional extensions (returns distribution, rolling correlation, smoothing) add context but are not substitutes for required tasks.

## 7. Reflection
### Challenges
- Dealing with very long historical data requires careful plotting choices to keep charts readable.
- Building a candlestick chart without specialized libraries requires custom drawing logic.
- Hypothesis tests can be sensitive to assumptions (independence, variance differences, non-normality).

### Solutions
- Used focused chart windows (e.g., last 180 days) for candlestick clarity.
- Combined rolling and convolution-based moving averages to validate trend extraction methods.
- Applied multiple normality checks (Q-Q plot and statistical tests) before interpreting inferential results.

### Next Improvements
- Add stationarity tests (ADF), autocorrelation diagnostics, and volatility models (e.g., GARCH).
- Compare AAPL behavior with market benchmarks for relative performance analysis.
- Extend to forecasting experiments with robust train/test time splits.